A LIDAR (light detection and ranging) measures the distance to surfaces by timing pulses of laser light. Here you compare pulsed and continuous-wave ranging, then look at the LIDARs that robots and cars carry, their specifications and the ways their readings go wrong. The [SLAM](/aiml-common/lectures/rse/slam/index#pillars-from-a-lidar-scan) section shows how a scan becomes landmarks: it splits the scan where the range jumps between neighboring beams and keeps the short clusters as pillars. The [beam model](/aiml-common/lectures/sensor-models/beam/index) section defines the likelihood of a scan given the robot's pose and the map, and estimates its parameters.

## Time of flight

A LIDAR sends a short pulse of infrared laser light and times its echo. The light travels to the surface and back, so a round trip of $\Delta t$ gives the range

$$
r = \frac{c\, \Delta t}{2},
$$

with $c$ the speed of light ([Li and Ibanez-Guzman 2020](https://arxiv.org/abs/2004.08467), Section II.A). A surface 6 m away returns the pulse after 40 ns. Measuring that delay precisely is difficult: a change of 1 cm in range changes the round trip by about 67 ps, so centimeter ranging needs timing electronics that resolve delays of that size (Corke, *Robotics, Vision and Control*, 3rd edition, Section 14.6.2).

In [ ]:
%config InlineBackend.figure_formats = ["svg"]

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Circle, Rectangle

plt.rcParams.update({"figure.figsize": (6.5, 3.5), "font.size": 9, "axes.titlesize": 10, "legend.fontsize": 8,
                     "svg.fonttype": "path", "axes.grid": False})
BLUE, VERMILLION, GREEN, PINK, GOLD, GRAY = "#0072B2", "#D55E00", "#009E73", "#cb77a6", "#c28600", "#7f7f7f"

In [ ]:
c = 299_792_458.0
r_target = 6.0
dt = 2 * r_target / c
t = np.linspace(-5e-9, 60e-9, 2000)
pulse = lambda t0, w=4e-9: np.exp(-0.5 * ((t - t0) / (w / 2.355)) ** 2)
fig, ax = plt.subplots(figsize=(6.5, 2.4))
ax.plot(t * 1e9, pulse(0), color=BLUE, label="emitted pulse")
ax.plot(t * 1e9, 0.35 * pulse(dt) + 0.02 * np.random.default_rng(0).normal(size=t.size), color=VERMILLION,
        label="echo from a surface at 6 m")
ax.annotate("", xy=(dt * 1e9, 1.08), xytext=(0, 1.08), arrowprops={"arrowstyle": "<->", "lw": 1})
ax.text(dt * 1e9 / 2, 1.13, rf"$\Delta t$ = {dt * 1e9:.0f} ns,  $r = c\,\Delta t / 2$ = 6 m", ha="center")
ax.set_xlabel("time (ns)")
ax.set_ylabel("signal (a.u.)")
ax.set_ylim(-0.1, 1.35)
ax.legend(loc="center right")
fig.tight_layout()
plt.show()

*Pulsed time of flight, drawn schematically. The echo from a surface at 6 m arrives 40 ns after the pulse leaves, weaker and in detector noise.*

Continuous-wave sensors measure the delay without timing a pulse. An amplitude-modulated continuous-wave (AMCW) sensor varies the brightness of a continuous beam and measures the range from the phase shift of the returned modulation ([Royo and Ballesta-Garcia 2019](https://doi.org/10.3390/app9194093), Section 2.1.2). A frequency-modulated continuous-wave (FMCW) sensor sweeps the optical frequency in a linear chirp and keeps a copy of it. Mixing the returning light with that copy produces a beat frequency. The delay sets the beat frequency, and a moving surface shifts it further by the Doppler effect. Sweeping the frequency up and then down separates the two, so one measurement gives both the range and the radial velocity of the surface ([Li and Ibanez-Guzman 2020](https://arxiv.org/abs/2004.08467), Section II.A.3).

Most LIDARs use near-infrared light between about 850 and 950 nm, which inexpensive silicon detectors can sense. At 1550 nm the eye-safety limits allow much more power, but the detectors must be made of InGaAs, which costs more, and water in the air absorbs more of the light ([Li and Ibanez-Guzman 2020](https://arxiv.org/abs/2004.08467), Section II.B.2; [Royo and Ballesta-Garcia 2019](https://doi.org/10.3390/app9194093), Section 3.1).

## 1D, 2D and 3D LIDARs

Corke describes three LIDAR configurations (*Robotics, Vision and Control*, Section 6.8). A 1D LIDAR, or laser rangefinder, measures a single range $r \in \mathbb R_{\ge 0}$ along its beam. A 2D LIDAR rotates that beam about an axis and returns a planar cross-section of the world in polar form, $\mathbb R_{\ge 0} \times S^1$. On a mobile robot it usually scans a plane parallel to the floor and slightly above it. A spinning 3D LIDAR rotates a vertical fan of beams and returns spherical coordinates in $\mathbb R_{\ge 0} \times S^2$, or points in $\mathbb R^3$, a *point cloud*. Other 3D LIDARs steer the beam without turning the whole head, as described below.

In [ ]:
def side_view(ax, elevations, title):
    floor, wall_x, h = 0.0, 4.0, 0.35  # sensor height 35 cm
    table = (1.6, 2.6, 0.72)  # x from, x to, top height
    ax.add_patch(Rectangle((-0.4, -0.08), 4.6, 0.08, color="#cccccc", lw=0))
    ax.plot([wall_x, wall_x], [0, 1.6], color="black", lw=2)
    ax.add_patch(Rectangle((table[0], table[2] - 0.04), table[1] - table[0], 0.04, color="#8c6d46", lw=0))
    for x in (table[0] + 0.05, table[1] - 0.05):  # legs at the corners, outside this slice: drawn dashed, not hit
        ax.plot([x, x], [0, table[2] - 0.04], color="#8c6d46", lw=1.2, ls="--")
    ax.add_patch(Rectangle((-0.3, 0.05), 0.4, 0.22, color="#555555", lw=0))  # robot base
    ax.add_patch(Rectangle((-0.16, 0.27), 0.12, 0.1, color="black", lw=0))  # sensor
    # surfaces in this vertical slice, as segments (x0, y0, x1, y1): floor, wall, table top's underside and top
    top, under = table[2], table[2] - 0.04
    segments = [(-0.4, floor, wall_x, floor), (wall_x, 0, wall_x, 1.6),
                (table[0], under, table[1], under), (table[0], top, table[1], top)]
    p0 = np.array([-0.1, h])
    for el in elevations:
        d = np.array([np.cos(np.deg2rad(el)), np.sin(np.deg2rad(el))])
        t = np.inf
        for x0, y0, x1, y1 in segments:  # first intersection of the ray with any segment
            e = np.array([x1 - x0, y1 - y0])
            den = d[0] * e[1] - d[1] * e[0]
            if abs(den) < 1e-12:
                continue
            w = np.array([x0, y0]) - p0
            ti = (w[0] * e[1] - w[1] * e[0]) / den
            ui = (w[0] * d[1] - w[1] * d[0]) / den
            if ti > 1e-9 and 0 <= ui <= 1:
                t = min(t, ti)
        q = p0 + t * d
        ax.plot(*np.c_[p0, q], color=VERMILLION, lw=0.9, alpha=0.9)
        ax.plot(*q, "o", color=VERMILLION, ms=3)
    ax.set_xlim(-0.5, 4.3)
    ax.set_ylim(-0.12, 1.7)
    ax.set_aspect("equal")
    ax.set_xlabel("distance (m)")
    ax.set_title(title)
    ax.set_yticks([0, 0.5, 1.0, 1.5])


fig, (a1, a2) = plt.subplots(2, 1, figsize=(6.5, 4.6), sharex=True)
side_view(a1, [0.0], "2D LIDAR: one scan plane at sensor height")
side_view(a2, np.linspace(-15, 15, 16), "3D LIDAR: 16 beams from -15° to +15° elevation")
a1.set_ylabel("height (m)")
a2.set_ylabel("height (m)")
a1.set_xlabel("")
a1.annotate("table top not seen", (2.1, 0.72), xytext=(2.1, 1.2), ha="center",
            arrowprops={"arrowstyle": "->", "lw": 0.8})
fig.tight_layout()
plt.show()

*A vertical slice through the middle of a table; the legs stand at the corners, outside the slice, and are drawn dashed. A 2D LIDAR scans one plane at its own height, passes under the table top and misses it. A 3D LIDAR with 16 beams between −15° and +15° elevation sees the floor, the underside of the table top and the wall.*

A 2D LIDAR sees nothing above or below its scan plane, such as a table top or a low curb. The TurtleBot3 carries one, the ROBOTIS LDS-01 or LDS-02. Industrial mobile robots use faster 2D scanners with a longer range, such as the 40 Hz Hokuyo UST-10LX. Research cars and mapping vehicles have long carried spinning 3D LIDARs with 16 to 128 beams, and newer designs also use the scanners without a spinning head described below.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/48/TurtleBot3_Burger.jpg/960px-TurtleBot3_Burger.jpg" alt="A TurtleBot3 Burger robot with its round 2D laser distance sensor mounted on top." width="55%" />
*TurtleBot3 Burger with its 360° laser distance sensor on top. Photo: Kuscu0, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:TurtleBot3_Burger.jpg).*

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/fe/Velodyne_High-Def_LIDAR.jpg/960px-Velodyne_High-Def_LIDAR.jpg" alt="A spinning Velodyne LIDAR with 64 laser beams mounted on the roof of Stanford's autonomous car Junior." width="75%" />
*A 64-beam spinning Velodyne LIDAR on the roof of Stanford's autonomous car Junior. Photo: Steve Jurvetson, [CC BY 2.0](https://creativecommons.org/licenses/by/2.0/), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Velodyne_High-Def_LIDAR.jpg).*

A spinning LIDAR has a motor and moving optics. In a *solid-state* LIDAR the scanning system has no moving parts. A MEMS LIDAR steers the beam with a tiny vibrating mirror and is called near-solid-state. Flash LIDARs, which light the whole scene at once, and optical phased arrays, which steer the beam by interference, are truly solid-state ([Li and Ibanez-Guzman 2020](https://arxiv.org/abs/2004.08467), Section II.C).

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/6/6e/Velodyne_Lidar_Velarray_H800_Velabit_Velarray_M1600_Sensor_Family.jpg/960px-Velodyne_Lidar_Velarray_H800_Velabit_Velarray_M1600_Sensor_Family.jpg" alt="Three compact box-shaped Velodyne solid-state LIDARs of different sizes." width="75%" />
*Velodyne's solid-state LIDARs: Velarray H800, Velabit and Velarray M1600. Photo: APJarvis, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Velodyne_Lidar_Velarray_H800_Velabit_Velarray_M1600_Sensor_Family.jpg).*

A spinning LIDAR traces the same lines on every turn, so gaps between the beams stay gaps. Some sensors trace a pattern that does not repeat. Livox says its Mid-360 works without rotating the transmitter and receiver, and that its coverage grows with time: after 0.1 s it is similar to that of a 32-line spinning LIDAR, and after 0.5 s it approaches 70% of the field of view ([Livox Mid-360 user manual](https://terra-1-g.djicdn.com/851d20f7b9f64838a34cd02351370894/Livox/Livox_Mid-360_User_Manual_EN.pdf)). One way to get such a pattern, sketched below, is a pair of prisms rotating at different rates, a *Risley prism* scanner. The sketch is a generic example, not the mechanism of the Mid-360 ([Raj et al. 2020](https://doi.org/10.3390/electronics9050741), Section 3.1).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.5, 3.0))
az = np.linspace(-35, 35, 140)
for el in np.linspace(-15, 15, 16):
    a1.plot(az, np.full_like(az, el), ".", color=BLUE, ms=1.5)
a1.set_title("spinning, 16 channels:\nthe same 16 lines every turn")
# a rosette from two counter-rotating prisms (Risley pair): the sum of two rotating unit vectors.
# The rate ratio is not a simple fraction, so each pass of the pattern lands between the previous ones.
f1, f2 = 1.0, -0.6180  # rotation rates in turns per unit time
for k, color in enumerate((GOLD, BLUE)):
    tt = np.linspace(k * 10.0, (k + 1) * 10.0, 4000)
    z = 0.5 * np.exp(2j * np.pi * f1 * tt) + 0.5 * np.exp(2j * np.pi * f2 * tt)
    a2.plot(35 * z.real, 35 * z.imag, color=color, lw=0.6, label=["first interval", "next interval"][k])
a2.legend(loc="lower right", fontsize=7)
a2.set_title("rotating prisms (sketch):\nlater intervals fill other gaps")
for ax in (a1, a2):
    ax.set_xlim(-38, 38)
    ax.set_ylim(-38, 38)
    ax.set_aspect("equal")
    ax.set_xlabel("azimuth (°)")
a1.set_ylabel("elevation (°)")
fig.tight_layout()
plt.show()

*Scan patterns in azimuth and elevation. A spinning LIDAR repeats its 16 lines. The right panel is a sketch, the sum of two unit vectors rotating at rates whose ratio is not a simple fraction. The rosette it traces never repeats, so each new time interval covers parts of the field that earlier intervals missed.*

The two tables below list sensors from the TurtleBot's to an automotive one, with the values their makers state. The first gives the geometry of each scan, the second how far, how often and how accurately each sensor measures. Each value is quoted with the maker's own term, and the conditions differ from one datasheet to the next.

| Sensor | Type | Field of view | Angular resolution |
|---|---|---|---|
| [ROBOTIS LDS-01](https://emanual.robotis.com/docs/en/platform/turtlebot3/appendix_lds_01/) | 2D | 360° | 1° |
| [ROBOTIS LDS-02](https://emanual.robotis.com/docs/en/platform/turtlebot3/appendix_lds_02/) | 2D | 360° | 1° |
| [Hokuyo UST-10LX](https://www.ai.uni-bremen.de/wiki/_media/software/ust-10lx_specification.pdf) | 2D | 270° | 0.25° |
| [Velodyne VLP-16](https://ouster.com/downloads/velodyne-downloads) | 3D, 16 beams | 360° × 30° | 0.1° to 0.4°, 2° between beams |
| [Ouster OS1, Rev 8](https://data.ouster.io/downloads/datasheets/datasheet-rev8-v4p0-os1.pdf) | 3D, 32 to 128 beams | 360° × 44° | 0.088° to 0.7° |
| [Livox Mid-360](https://www.livoxtech.com/mid-360/specs) | 3D, non-repeating pattern | 360° × 59° | no fixed grid |

| Sensor | Range | Scan or frame rate | Range error, as stated |
|---|---|---|---|
| ROBOTIS LDS-01 | 0.12 to 3.5 m | 5 Hz | accuracy ±15 mm below 0.5 m, ±5% beyond |
| ROBOTIS LDS-02 | 0.16 to 8 m | 5 Hz or more | accuracy ±10 mm below 0.3 m, ±3% to 6 m; precision ±5% from 6 to 8 m |
| Hokuyo UST-10LX | 10 m on white, 4 m at 10% reflectance | 40 Hz | accuracy ±40 mm, repeatability σ < 30 mm |
| Velodyne VLP-16 | 100 m | 5 to 20 Hz | up to ±3 cm (typical) |
| Ouster OS1, Rev 8 | 90 m at 10%, 170 m at 80% reflectivity | 5 to 40 Hz | accuracy ±1.25 cm on diffuse targets |
| Livox Mid-360 | 40 m at 10%, 70 m at 80% reflectivity | 10 Hz frames | precision 1σ ≤ 2 cm at 10 m |

The Hokuyo ranges are the guaranteed detection ranges, on a white Kent sheet and on a 10% diffuse target; it outputs readings out to 30 m, and its error figures hold under the factory test conditions. The Livox ranges are stated for 100 klx of ambient light, about full sunlight. A Livox frame is a 0.1 s slice of a pattern that never repeats, not a full revolution.

Compared with a camera, a LIDAR returns metric range directly and works in the dark. Sunlight adds infrared background at the sensor's wavelength, which can drown a weak echo and shorten the range. A sweep takes time, so a moving sensor distorts its own scan. A LIDAR measures no color, and a spinning one needs a motor. Compared with a typical camera, a typical robot LIDAR is larger, draws more power and costs more (Corke, Section 6.8).

## Intensity and failure modes

Most scanning LIDARs also report, with each range, how strongly the light came back, the *intensity* or *remission*. It depends on the infrared reflectivity of the surface (Corke, Section 6.8) and on the angle at which the beam strikes it ([Li and Ibanez-Guzman 2020](https://arxiv.org/abs/2004.08467), Section II.A.1). A dark surface returns less light, so the sensor sees it over a shorter distance. The range table above shows this: the OS1 reaches 170 m on an 80% reflective target and 90 m on a 10% one. A black, light-absorbing object may return nothing at all (*Probabilistic Robotics*, Section 6.3.1).

Glass passes most of the beam and reflects the rest like a mirror, so a scanner detects a pane only over a limited range of incidence angles. That range narrows when a wall stands behind the glass ([Foster et al. 2013](https://web.eecs.umich.edu/~kuipers/papers/Foster-icra-13.pdf)). A beam can also bounce off glass or a mirror onto another object and come back by way of the mirror. The sensor converts the longer travel time into a range along the beam's original direction, so it places a point on the far side of the reflecting surface, where nothing is. This is *multipath*, and it leaves false points in the scan ([Li, Zhao and Schwertfeger 2024](https://arxiv.org/abs/2406.10494), Section 4). Fog, rain, dust and snow scatter and absorb the light and produce false detections ([Royo and Ballesta-Garcia 2019](https://doi.org/10.3390/app9194093), Section 4.4). The beam spreads as it travels, so the spot it lights grows with distance. At the edge of an object the spot can fall partly on the object and partly on a wall behind it, and the pulse comes back as two echoes. The VLP-16 reports the strongest return, the last return or both, depending on its return mode. In dual-return mode it reports both echoes when the two surfaces are at least 1 m apart ([VLP-16 user manual](https://data.ouster.io/downloads/velodyne/user-manual/vlp-16-user-manual-revf.pdf), Chapter 6).

The [beam model](/aiml-common/lectures/sensor-models/beam/index) does not model each of these failures. It describes what the readings of one beam look like with four terms: a Gaussian around the expected range for a correct return, an exponential for returns from objects in front of the mapped surface, a point mass at the maximum range for a beam that returns nothing, and a uniform density for readings that the other three do not explain. A multipath point or a split return lands in whichever term its range falls under, most often the uniform one.
